In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import cv2
import numpy as np
import os
import shutil
from pathlib import Path

# 1. SETUP PATHS
input_base = '/kaggle/input/et-autism-dataset' 
output_base = '/kaggle/working/cropped_autism_dataset'
classes = ['high', 'low', 'medium', 'mild']

# 2. DEFINE THE IRIS-CENTERED CROP FUNCTION
def iris_crop(img_path, output_path, crop_size=224):
    img = cv2.imread(str(img_path))
    if img is None: return False
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    
    # Detect Iris using Hough Circles
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1, 50,
                               param1=50, param2=30, minRadius=30, maxRadius=150)
    
    if circles is not None:
        circles = np.uint16(np.around(circles))
        i = circles[0, 0] 
        center_x, center_y, radius = i[0], i[1], i[2]
        
        # Calculate tight crop boundaries
        margin = int(radius * 2.5)
        y1, y2 = max(0, center_y - margin), min(img.shape[0], center_y + margin)
        x1, x2 = max(0, center_x - margin), min(img.shape[1], center_x + margin)
        
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: return False
        
        resized = cv2.resize(crop, (crop_size, crop_size))
        cv2.imwrite(str(output_path), resized)
        return True
    return False

# 3. UPDATED LOOP FOR NESTED FOLDERS
failed_images = []

for cls in classes:
    # We point to the top level class folder
    input_class_path = Path(input_base) / cls
    output_class_path = Path(output_base) / cls
    output_class_path.mkdir(parents=True, exist_ok=True)
    
    # FIX: Use rglob or recursive search to find images in subfolders
    all_images = list(input_class_path.rglob('*.jpg'))
    print(f"\n--- Processing Class: {cls.upper()} ---")
    print(f"Found {len(all_images)} images inside nested folders. Starting crop...")
    
    count = 0
    for img_path in all_images:
        # Keep the filename but save it directly into the new class folder
        output_path = output_class_path / img_path.name
        if iris_crop(img_path, output_path):
            count += 1
        else:
            failed_images.append(str(img_path))
            
    print(f"Successfully saved {count} cropped images for {cls}.")

# 4. SUMMARY AND ZIP
print("\n--- PROCESSING SUMMARY ---")
print(f"Total Failures: {len(failed_images)}")

print("\n--- ZIPPING DATASET ---")
shutil.make_archive('cropped_dataset', 'zip', output_base)
print("Zip file created. Check the 'Output' section on the right panel.")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil
import os

# 1. SETUP PATHS
input_base = '/kaggle/input/et-autism-dataset' 
output_base = '/kaggle/working/cropped_autism_dataset'
classes = ['high', 'low', 'medium', 'mild']

# 2. LOAD DETECTOR
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

def resilient_crop(img_path, output_path, crop_size=224):
    img = cv2.imread(str(img_path))
    if img is None: return False
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Detect eye regions using pattern matching instead of circles
    eyes = eye_cascade.detectMultiScale(gray, 1.05, 4) # Fine-tuned scaleFactor

    if len(eyes) > 0:
        # Take the largest detected region
        (x, y, w, h) = sorted(eyes, key=lambda e: e[2], reverse=True)[0]
        
        # Sqr crop around the eye with a 30% safety margin
        margin = int(w * 0.3)
        y1, y2 = max(0, y-margin), min(img.shape[0], y+h+margin)
        x1, x2 = max(0, x-margin), min(img.shape[1], x+w+margin)
        
        crop = img[y1:y2, x1:x2]
    else:
        # FALLBACK: If AI fails, take the center 70% of the photo
        # This ensures we still capture the eye since it's usually centered
        h, w, _ = img.shape
        side = int(min(h, w) * 0.7)
        y1, x1 = (h - side) // 2, (w - side) // 2
        crop = img[y1:y1+side, x1:x1+side]

    resized = cv2.resize(crop, (crop_size, crop_size))
    cv2.imwrite(str(output_path), resized)
    return True

# 3. EXECUTE RECURSIVE LOOP
for cls in classes:
    input_class_path = Path(input_base) / cls
    output_class_path = Path(output_base) / cls
    output_class_path.mkdir(parents=True, exist_ok=True)
    
    all_images = list(input_class_path.rglob('*.jpg'))
    print(f"Processing {cls.upper()}: Found {len(all_images)} images.")
    
    count = 0
    for img_path in all_images:
        output_path = output_class_path / img_path.name
        if resilient_crop(img_path, output_path):
            count += 1
            
    print(f"Successfully processed {count} images for {cls}.")

# 4. CREATE DOWNLOADABLE ZIP
print("\n--- ZIPPING FOR DOWNLOAD ---")
shutil.make_archive('final_cropped_dataset', 'zip', output_base)
print("Done! Download 'final_cropped_dataset.zip' from the Output sidebar.")

In [ ]:
!pip install mediapipe


In [1]:

import cv2
import numpy as np
import os
import shutil
from pathlib import Path
import mediapipe as mp

# 1. SETUP PATHS
input_base = '/kaggle/input/et-autism-dataset' 
output_base = '/kaggle/working/cropped_autism_dataset'
classes = ['high', 'low', 'medium', 'mild']

# 2. INITIALIZE MEDIAPIPE IRIS TRACKER
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True, # Required for Iris landmarks
    min_detection_confidence=0.3 # Lowered for your blurry images
)

def smart_iris_crop(img_path, output_path, crop_size=224):
    img = cv2.imread(str(img_path))
    if img is None: return False
    
    # MediaPipe requires RGB
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_img)
    
    h, w, _ = img.shape
    
    if results.multi_face_landmarks:
        # Landmark 468 is the center of the left iris
        # We calculate the exact pixel location
        landmark = results.multi_face_landmarks[0].landmark[468]
        center_x, center_y = int(landmark.x * w), int(landmark.y * h)
        
        # FINE-TUNED CROP: 30% of image size around the iris center
        # This removes 70% of the face while keeping the eye perfectly centered
        side = int(w * 0.15) 
        y1, y2 = max(0, center_y - side), min(h, center_y + side)
        x1, x2 = max(0, center_x - side), min(w, center_x + side)
        
        crop = img[y1:y2, x1:x2]
    else:
        # FALLBACK: If AI fails, use a tight center crop
        side = int(min(h, w) * 0.3)
        y1, x1 = (h - side) // 2, (w - side) // 2
        crop = img[y1:y1+side, x1:x1+side]

    if crop.size == 0: return False
    
    resized = cv2.resize(crop, (crop_size, crop_size))
    cv2.imwrite(str(output_path), resized)
    return True

# 3. RUN THE LOOP FOR ALL 4000 IMAGES
for cls in classes:
    input_class_path = Path(input_base) / cls
    output_class_path = Path(output_base) / cls
    output_class_path.mkdir(parents=True, exist_ok=True)
    
    all_images = list(input_class_path.rglob('*.jpg'))
    print(f"Processing {cls.upper()}: {len(all_images)} images found.")
    
    count = 0
    for img_path in all_images:
        output_path = output_class_path / img_path.name
        if smart_iris_crop(img_path, output_path):
            count += 1
            
    print(f"Successfully processed {count} images for {cls}.")

# 4. ZIP AND DOWNLOAD
shutil.make_archive('smart_cropped_dataset', 'zip', output_base)
print("\n--- DONE ---")
print("Download 'smart_cropped_dataset.zip' from the Output sidebar.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.11/dist-package

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [1]:
import cv2
import numpy as np
import os
import shutil
from pathlib import Path

# 1. SETUP PATHS
input_base = '/kaggle/input/et-autism-dataset' 
output_base = '/kaggle/working/cropped_autism_dataset'
classes = ['high', 'low', 'medium', 'mild']

# 2. ROBUST PUPIL-CENTERED CROP FUNCTION
def robust_iris_crop(img_path, output_path, crop_size=224):
    img = cv2.imread(str(img_path))
    if img is None: return False
    
    # Pre-processing: Blur to remove tiny dark noise pixels
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (15, 15), 0) # Increased blur for stability
    
    # Locate the darkest point (The Pupil)
    _, _, min_loc, _ = cv2.minMaxLoc(blurred)
    center_x, center_y = min_loc
    
    # FINE-TUNED PARAMETER: Crop size
    # We take 55% of the image width to ensure we get the full iris but no skin
    h, w, _ = img.shape
    side = int(w * 0.42) 
    
    # Define boundaries with boundary checking
    y1, y2 = max(0, center_y - side), min(h, center_y + side)
    x1, x2 = max(0, center_x - side), min(w, center_x + side)
    
    crop = img[y1:y2, x1:x2]
    
    # CORRUPTION CHECK: Ensure the crop didn't fail or result in a tiny sliver
    if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
        # Emergency Fallback: Center crop if pupil detection is wild
        s = int(min(h, w) * 0.4)
        crop = img[(h//2)-s:(h//2)+s, (w//2)-s:(w//2)+s]
    
    resized = cv2.resize(crop, (crop_size, crop_size))
    cv2.imwrite(str(output_path), resized)
    return True

# 3. DATASET LOOP (Handles nested folders)
for cls in classes:
    input_class_path = Path(input_base) / cls
    output_class_path = Path(output_base) / cls
    output_class_path.mkdir(parents=True, exist_ok=True)
    
    all_images = list(input_class_path.rglob('*.jpg'))
    print(f"Processing {cls.upper()}: Found {len(all_images)} images.")
    
    count = 0
    for img_path in all_images:
        output_path = output_class_path / img_path.name
        if robust_iris_crop(img_path, output_path):
            count += 1
            
    print(f"Successfully saved {count} cropped images for {cls}.")

# 4. DOWNLOAD ZIP
print("\n--- ZIPPING DATASET ---")
shutil.make_archive('robust_cropped_dataset_final_2nd', 'zip', output_base)
print("Done! Download 'robust_cropped_dataset.zip' from the Output panel.")

Processing HIGH: Found 1000 images.
Successfully saved 1000 cropped images for high.
Processing LOW: Found 1000 images.
Successfully saved 1000 cropped images for low.
Processing MEDIUM: Found 1000 images.
Successfully saved 1000 cropped images for medium.
Processing MILD: Found 1000 images.
Successfully saved 1000 cropped images for mild.

--- ZIPPING DATASET ---
Done! Download 'robust_cropped_dataset.zip' from the Output panel.
